In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import mnist
import os

tf.keras.utils.disable_interactive_logging()

print("--- Stage 1: Preparing MNIST Data ---")

(digits_train, _), (_, _) = mnist.load_data()
digits_train = digits_train.reshape(digits_train.shape[0], 28, 28, 1).astype('float32')
digits_train = (digits_train - 127.5) / 127.5
print("Scaled training tensor shape:", digits_train.shape)
print("Sample scaled pixel value:", digits_train[10, 14, 14, 0])

noise_dim = 100

def make_generator():
    net = keras.Sequential(name="digit_generator")
    net.add(layers.Dense(7 * 7 * 256, use_bias=False, input_shape=(noise_dim,)))
    net.add(layers.BatchNormalization())
    net.add(layers.LeakyReLU(alpha=0.2))
    net.add(layers.Reshape((7, 7, 256)))

    net.add(layers.Conv2DTranspose(128, (5, 5), strides=(1, 1), padding='same', use_bias=False))
    net.add(layers.BatchNormalization())
    net.add(layers.LeakyReLU(alpha=0.2))

    net.add(layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False))
    net.add(layers.BatchNormalization())
    net.add(layers.LeakyReLU(alpha=0.2))

    net.add(layers.Conv2DTranspose(32, (5, 5), strides=(1, 1), padding='same', use_bias=False))
    net.add(layers.BatchNormalization())
    net.add(layers.LeakyReLU(alpha=0.2))

    net.add(layers.Conv2DTranspose(1, (5, 5), strides=(2, 2), padding='same', use_bias=False, activation='tanh'))
    return net

gen_model = make_generator()
print("\n--- Generator Architecture ---")
gen_model.summary()

def make_discriminator():
    net = keras.Sequential(name="digit_discriminator")
    net.add(layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same', input_shape=[28, 28, 1]))
    net.add(layers.LeakyReLU(alpha=0.2))
    net.add(layers.Dropout(0.3))

    net.add(layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same'))
    net.add(layers.LeakyReLU(alpha=0.2))
    net.add(layers.Dropout(0.3))

    net.add(layers.Flatten())
    net.add(layers.Dense(1, activation='sigmoid'))
    return net

disc_model = make_discriminator()
print("\n--- Discriminator Architecture ---")
disc_model.summary()

bce_loss = keras.losses.BinaryCrossentropy(from_logits=False)

def calc_discriminator_loss(real_scores, fake_scores):
    real_term = bce_loss(tf.ones_like(real_scores), real_scores)
    fake_term = bce_loss(tf.zeros_like(fake_scores), fake_scores)
    return real_term + fake_term

def calc_generator_loss(fake_scores):
    return bce_loss(tf.ones_like(fake_scores), fake_scores)

gen_opt = tf.keras.optimizers.Adam(learning_rate=2e-4, beta_1=0.5)
disc_opt = tf.keras.optimizers.Adam(learning_rate=2e-4, beta_1=0.5)

BATCH_SIZE = 128
NUM_EPOCHS = 150
NUM_PREVIEW_IMAGES = 16
NOISE_DIM = noise_dim

preview_seed = tf.random.normal([NUM_PREVIEW_IMAGES, NOISE_DIM])

@tf.function
def run_train_step(real_images):
    random_noise = tf.random.normal([BATCH_SIZE, NOISE_DIM])

    with tf.GradientTape() as g_tape, tf.GradientTape() as d_tape:
        fake_images = gen_model(random_noise, training=True)
        real_scores = disc_model(real_images, training=True)
        fake_scores = disc_model(fake_images, training=True)

        g_loss = calc_generator_loss(fake_scores)
        d_loss = calc_discriminator_loss(real_scores, fake_scores)

    g_grads = g_tape.gradient(g_loss, gen_model.trainable_variables)
    d_grads = d_tape.gradient(d_loss, disc_model.trainable_variables)

    gen_opt.apply_gradients(zip(g_grads, gen_model.trainable_variables))
    disc_opt.apply_gradients(zip(d_grads, disc_model.trainable_variables))

    return g_loss, d_loss

def preview_generated_digits(model, epoch_num, seed_input):
    samples = model(seed_input, training=False)
    samples = (samples * 0.5) + 0.5

    plt.figure(figsize=(4, 4))
    for i in range(samples.shape[0]):
        plt.subplot(4, 4, i + 1)
        plt.imshow(samples[i, :, :, 0], cmap='gray')
        plt.axis('off')
    plt.suptitle(f"Generated Digits — Epoch {epoch_num}", fontsize=16)

    if not os.path.exists('gan_outputs'):
        os.makedirs('gan_outputs')
    plt.savefig(f'gan_outputs/digits_epoch_{epoch_num:04d}.png')
    plt.show()

mnist_dataset = tf.data.Dataset.from_tensor_slices(digits_train).shuffle(digits_train.shape[0]).batch(BATCH_SIZE)

gen_loss_history = []
disc_loss_history = []

def run_training(dataset, num_epochs):
    print("\n--- Starting GAN Training ---")
    for epoch in range(num_epochs):
        epoch_gen_losses = []
        epoch_disc_losses = []

        for image_batch in dataset:
            g_loss, d_loss = run_train_step(image_batch)
            epoch_gen_losses.append(g_loss.numpy())
            epoch_disc_losses.append(d_loss.numpy())

        mean_gen_loss = np.mean(epoch_gen_losses)
        mean_disc_loss = np.mean(epoch_disc_losses)
        gen_loss_history.append(mean_gen_loss)
        disc_loss_history.append(mean_disc_loss)

        print(f"Epoch {epoch + 1}/{num_epochs} | Gen Loss: {mean_gen_loss:.4f} | Disc Loss: {mean_disc_loss:.4f}")

        if (epoch + 1) % 15 == 0:
            preview_generated_digits(gen_model, epoch + 1, preview_seed)

    print("\n--- Training Finished. Generating Final Sample ---")
    preview_generated_digits(gen_model, num_epochs, preview_seed)

run_training(mnist_dataset, NUM_EPOCHS)

plt.figure(figsize=(10, 5))
plt.plot(gen_loss_history, label='Generator Loss')
plt.plot(disc_loss_history, label='Discriminator Loss')
plt.title('GAN Training Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

--- Stage 1: Preparing MNIST Data ---
Scaled training tensor shape: (60000, 28, 28, 1)
Sample scaled pixel value: 0.99215686


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(



--- Generator Architecture ---



--- Discriminator Architecture ---


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



--- Starting GAN Training ---
Epoch 1/150 | Gen Loss: 0.7840 | Disc Loss: 1.2279
Epoch 2/150 | Gen Loss: 0.7665 | Disc Loss: 1.3256
Epoch 3/150 | Gen Loss: 0.7823 | Disc Loss: 1.3130
Epoch 4/150 | Gen Loss: 0.7550 | Disc Loss: 1.3380
Epoch 5/150 | Gen Loss: 0.7323 | Disc Loss: 1.3601
Epoch 6/150 | Gen Loss: 0.7325 | Disc Loss: 1.3570
Epoch 7/150 | Gen Loss: 0.7339 | Disc Loss: 1.3535
Epoch 8/150 | Gen Loss: 0.7420 | Disc Loss: 1.3464
Epoch 9/150 | Gen Loss: 0.7484 | Disc Loss: 1.3414
